In [4]:
import torch
import glob
import os
from tqdm import tqdm
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support
)
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

# This assumes 'utils.py' is in the same directory
from utils import SegmentFromFile

def evaluate_model(model_path, validation_data_directory, time_gap, device="cuda"):
    """
    Evaluates a fine-tuned Hugging Face classifier model on validation data.

    This function is adapted for models trained with the specific "T{num} G{num}"
    text format and does not use quantization or PEFT adapters.

    Args:
        model_path (str): Path to the directory containing the fine-tuned model.
        validation_data_directory (str): Path to the directory of validation CSV files.
        time_gap (float): The time gap used for segmenting the validation data.
        device (str): The device to run inference on (e.g., "cuda" or "cpu").

    Returns:
        dict: A dictionary containing performance metrics (accuracy, f1, precision, recall).
    """
    # --- 1. Load the Fine-Tuned Model and Tokenizer ---
    print(f"\n--- Starting Evaluation of Model: {model_path} ---")
    if not os.path.isdir(model_path):
        raise FileNotFoundError(f"Model directory not found at: {model_path}")

    print("Loading fine-tuned model and tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    model.to(device)
    model.eval()

    # --- 2. Load and Process Validation Data ---
    print(f"Loading validation data from: {validation_data_directory}")
    all_chunks = []
    all_labels = []

    csv_files = glob.glob(os.path.join(validation_data_directory, "*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in: {validation_data_directory}")

    for file_path in tqdm(csv_files, desc="Processing validation files"):
        filename = os.path.basename(file_path)
        chunks, labels = SegmentFromFile(validation_data_directory, filename, time_gap=time_gap)
        all_chunks.extend(chunks)
        all_labels.extend(labels)

    print(f"Total validation segments loaded: {len(all_chunks)}")

    # --- 3. Format Data to Match Training Input ---
    # This MUST match the format used in the training script.
    def format_chunk_to_string(chunk):
        tokens = []
        for pair in chunk:
            token1 = f"T{int(pair[0])}"
            token2 = f"G{int(pair[1])}"
            tokens.append(token1)
            tokens.append(token2)
        return " ".join(tokens)

    texts = [format_chunk_to_string(chunk) for chunk in tqdm(all_chunks, desc="Formatting segments")]

    # --- 4. Run Inference in Batches ---
    print("Running inference on validation data...")
    batch_size = 16  # Adjust based on your GPU memory
    all_preds = []
    all_probs = []

    num_batches = (len(texts) + batch_size - 1) // batch_size
    for i in tqdm(range(0, len(texts), batch_size), total=num_batches, desc="Inference"):
        batch_texts = texts[i:i + batch_size]
        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
            probs = torch.softmax(logits, dim=-1)
            preds = torch.argmax(logits, dim=-1)

        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs[:, 1].cpu().numpy()) # Probability of the 'attack' class (class 1)

    # --- 5. Calculate and Display Metrics ---
    print("\nCalculating metrics...")
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='binary')
    acc = accuracy_score(all_labels, all_preds)
    cm = confusion_matrix(all_labels, all_preds)

    metrics = {'accuracy': acc, 'f1': f1, 'precision': precision, 'recall': recall}

    print("\n=== Validation Results ===")
    print(f"Accuracy:  {acc:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")

    print("\n=== Confusion Matrix ===")
    tn, fp, fn, tp = cm.ravel()
    print(f"               Predicted Negative | Predicted Positive")
    print(f"Actual Negative: {tn:<17d}| {fp:<17d}")
    print(f"Actual Positive: {fn:<17d}| {tp:<17d}")
    print("=" * 30)

    return metrics

/home/lisa/Arupreza/UIDS-II/uids/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# --- 10. Evaluate a Specific Checkpoint ---
# The path you provided is used here
TIME_GAP_TEST = 83.0
VALIDATION_DATA_DIRECTORY = "/home/lisa/Arupreza/UIDS-II/Split_data/Test/Tesla"
checkpoint_to_evaluate = "/home/lisa/Arupreza/UIDS-II/SFTSrc/bert/TrainKia/mobilebert-can-attack-classifier-experimental"

print(f"\n--- Evaluating specific checkpoint: {checkpoint_to_evaluate} ---")
evaluate_model(
    model_path=checkpoint_to_evaluate,
    validation_data_directory=VALIDATION_DATA_DIRECTORY, # From your script's config
    time_gap=TIME_GAP_TEST                             # From your script's config
)


--- Evaluating specific checkpoint: /home/lisa/Arupreza/UIDS-II/SFTSrc/bert/TrainKia/mobilebert-can-attack-classifier-experimental ---

--- Starting Evaluation of Model: /home/lisa/Arupreza/UIDS-II/SFTSrc/bert/TrainKia/mobilebert-can-attack-classifier-experimental ---
Loading fine-tuned model and tokenizer...
Loading validation data from: /home/lisa/Arupreza/UIDS-II/Split_data/Test/Tesla


Processing validation files: 100%|██████████| 16/16 [00:07<00:00,  2.02it/s]


Total validation segments loaded: 7514


Formatting segments: 100%|██████████| 7514/7514 [00:00<00:00, 8274.82it/s]


Running inference on validation data...


Inference: 100%|██████████| 470/470 [00:39<00:00, 11.95it/s]


Calculating metrics...

=== Validation Results ===
Accuracy:  0.9570
F1 Score:  0.9697
Precision: 0.9479
Recall:    0.9925

=== Confusion Matrix ===
               Predicted Negative | Predicted Positive
Actual Negative: 2028             | 284              
Actual Positive: 39               | 5163             


{'accuracy': 0.9570135746606335,
 'f1': 0.9696685134754437,
 'precision': 0.9478612080044061,
 'recall': 0.9925028835063437}

In [34]:
# --- 10. Evaluate a Specific Checkpoint ---
# The path you provided is used here
TIME_GAP_TEST = 100.0
VALIDATION_DATA_DIRECTORY = "/home/lisa/Arupreza/UIDS-II/Split_data/Test/Sil"
checkpoint_to_evaluate = "/home/lisa/Arupreza/UIDS-II/SFTSrc/bert/TrainKia/mobilebert-can-attack-classifier-experimental"

print(f"\n--- Evaluating specific checkpoint: {checkpoint_to_evaluate} ---")
evaluate_model(
    model_path=checkpoint_to_evaluate,
    validation_data_directory=VALIDATION_DATA_DIRECTORY, # From your script's config
    time_gap=TIME_GAP_TEST                             # From your script's config
)


--- Evaluating specific checkpoint: /home/lisa/Arupreza/UIDS-II/SFTSrc/bert/TrainKia/mobilebert-can-attack-classifier-experimental ---

--- Starting Evaluation of Model: /home/lisa/Arupreza/UIDS-II/SFTSrc/bert/TrainKia/mobilebert-can-attack-classifier-experimental ---
Loading fine-tuned model and tokenizer...
Loading validation data from: /home/lisa/Arupreza/UIDS-II/Split_data/Test/Sil


Processing validation files: 100%|██████████| 16/16 [00:07<00:00,  2.02it/s]


Total validation segments loaded: 7595


Formatting segments: 100%|██████████| 7595/7595 [00:00<00:00, 8663.78it/s]


Running inference on validation data...


Inference: 100%|██████████| 475/475 [00:40<00:00, 11.85it/s]


Calculating metrics...

=== Validation Results ===
Accuracy:  0.9771
F1 Score:  0.9836
Precision: 0.9760
Recall:    0.9913

=== Confusion Matrix ===
               Predicted Negative | Predicted Positive
Actual Negative: 2209             | 128              
Actual Positive: 46               | 5212             


{'accuracy': 0.9770901909150758,
 'f1': 0.9835818078882808,
 'precision': 0.9760299625468165,
 'recall': 0.9912514263978699}

In [4]:
# --- 10. Evaluate a Specific Checkpoint ---
# The path you provided is used here
TIME_GAP_TEST = 105.0
VALIDATION_DATA_DIRECTORY = "/home/lisa/Arupreza/UIDS-II/Split_data/Test/Gen"
checkpoint_to_evaluate = "/home/lisa/Arupreza/UIDS-II/SFTSrc/bert/TrainKia/mobilebert-can-attack-classifier-experimental"

print(f"\n--- Evaluating specific checkpoint: {checkpoint_to_evaluate} ---")
evaluate_model(
    model_path=checkpoint_to_evaluate,
    validation_data_directory=VALIDATION_DATA_DIRECTORY, # From your script's config
    time_gap=TIME_GAP_TEST                             # From your script's config
)


--- Evaluating specific checkpoint: /home/lisa/Arupreza/UIDS-II/SFTSrc/bert/TrainKia/mobilebert-can-attack-classifier-experimental ---

--- Starting Evaluation of Model: /home/lisa/Arupreza/UIDS-II/SFTSrc/bert/TrainKia/mobilebert-can-attack-classifier-experimental ---
Loading fine-tuned model and tokenizer...
Loading validation data from: /home/lisa/Arupreza/UIDS-II/Split_data/Test/Gen


Processing validation files:  56%|█████▋    | 9/16 [00:06<00:05,  1.30it/s]/home/lisa/Arupreza/UIDS-II/SFTSrc/utils.py:91: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
Processing validation files:  69%|██████▉   | 11/16 [00:08<00:03,  1.48it/s]/home/lisa/Arupreza/UIDS-II/SFTSrc/utils.py:91: DtypeWarning: Columns (20,22) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
Processing validation files: 100%|██████████| 16/16 [00:14<00:00,  1.11it/s]


Total validation segments loaded: 7368


Formatting segments: 100%|██████████| 7368/7368 [00:00<00:00, 8094.35it/s]


Running inference on validation data...


Inference: 100%|██████████| 461/461 [00:38<00:00, 11.93it/s]


Calculating metrics...

=== Validation Results ===
Accuracy:  0.9868
F1 Score:  0.9905
Precision: 0.9892
Recall:    0.9918

=== Confusion Matrix ===
               Predicted Negative | Predicted Positive
Actual Negative: 2213             | 55               
Actual Positive: 42               | 5058             


{'accuracy': 0.9868349619978285,
 'f1': 0.9905023009889357,
 'precision': 0.9892431058087229,
 'recall': 0.991764705882353}

In [36]:
# --- 10. Evaluate a Specific Checkpoint ---
# The path you provided is used here
TIME_GAP_TEST = 100.0
VALIDATION_DATA_DIRECTORY = "/home/lisa/Arupreza/UIDS-II/Split_data/Test/Low"
checkpoint_to_evaluate = "/home/lisa/Arupreza/UIDS-II/SFTSrc/bert/TrainKia/mobilebert-can-attack-classifier-experimental"

print(f"\n--- Evaluating specific checkpoint: {checkpoint_to_evaluate} ---")
evaluate_model(
    model_path=checkpoint_to_evaluate,
    validation_data_directory=VALIDATION_DATA_DIRECTORY, # From your script's config
    time_gap=TIME_GAP_TEST                             # From your script's config
)


--- Evaluating specific checkpoint: /home/lisa/Arupreza/UIDS-II/SFTSrc/bert/TrainKia/mobilebert-can-attack-classifier-experimental ---

--- Starting Evaluation of Model: /home/lisa/Arupreza/UIDS-II/SFTSrc/bert/TrainKia/mobilebert-can-attack-classifier-experimental ---
Loading fine-tuned model and tokenizer...
Loading validation data from: /home/lisa/Arupreza/UIDS-II/Split_data/Test/Low


Processing validation files:  56%|█████▋    | 9/16 [00:05<00:06,  1.09it/s]/home/lisa/Arupreza/UIDS-II/SFTSrc/utils.py:91: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
Processing validation files:  62%|██████▎   | 10/16 [00:06<00:04,  1.23it/s]/home/lisa/Arupreza/UIDS-II/SFTSrc/utils.py:91: DtypeWarning: Columns (20,22) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
Processing validation files: 100%|██████████| 16/16 [00:16<00:00,  1.05s/it]


Total validation segments loaded: 12929


Formatting segments: 100%|██████████| 12929/12929 [00:01<00:00, 8617.16it/s]


Running inference on validation data...


Inference: 100%|██████████| 809/809 [01:08<00:00, 11.84it/s]


Calculating metrics...

=== Validation Results ===
Accuracy:  0.9501
F1 Score:  0.9221
Precision: 0.8944
Recall:    0.9517

=== Confusion Matrix ===
               Predicted Negative | Predicted Positive
Actual Negative: 8464             | 451              
Actual Positive: 194              | 3820             


{'accuracy': 0.9501121509784206,
 'f1': 0.9221484610742305,
 'precision': 0.8944041208147975,
 'recall': 0.9516691579471849}

In [6]:
# --- 10. Evaluate a Specific Checkpoint ---
# The path you provided is used here
TIME_GAP_TEST = 125.0
VALIDATION_DATA_DIRECTORY = "/home/lisa/Arupreza/UIDS-II/OtherLAB/Kia"
checkpoint_to_evaluate = "/home/lisa/Arupreza/UIDS-II/SFTSrc/bert/TrainKia/mobilebert-can-attack-classifier-experimental"

print(f"\n--- Evaluating specific checkpoint: {checkpoint_to_evaluate} ---")
evaluate_model(
    model_path=checkpoint_to_evaluate,
    validation_data_directory=VALIDATION_DATA_DIRECTORY, # From your script's config
    time_gap=TIME_GAP_TEST                             # From your script's config
)


--- Evaluating specific checkpoint: /home/lisa/Arupreza/UIDS-II/SFTSrc/bert/TrainKia/mobilebert-can-attack-classifier-experimental ---

--- Starting Evaluation of Model: /home/lisa/Arupreza/UIDS-II/SFTSrc/bert/TrainKia/mobilebert-can-attack-classifier-experimental ---
Loading fine-tuned model and tokenizer...
Loading validation data from: /home/lisa/Arupreza/UIDS-II/OtherLAB/Kia


Processing validation files: 100%|██████████| 4/4 [00:48<00:00, 12.07s/it]


Total validation segments loaded: 4231


Formatting segments: 100%|██████████| 4231/4231 [00:00<00:00, 7025.04it/s]


Running inference on validation data...


Inference: 100%|██████████| 265/265 [00:22<00:00, 11.93it/s]


Calculating metrics...

=== Validation Results ===
Accuracy:  0.9813
F1 Score:  0.9760
Precision: 0.9994
Recall:    0.9536

=== Confusion Matrix ===
               Predicted Negative | Predicted Positive
Actual Negative: 2548             | 1                
Actual Positive: 78               | 1604             


{'accuracy': 0.9813282911841172,
 'f1': 0.9759659263766353,
 'precision': 0.9993769470404984,
 'recall': 0.9536266349583828}

In [7]:
# --- 10. Evaluate a Specific Checkpoint ---
# The path you provided is used here
TIME_GAP_TEST = 130.0
VALIDATION_DATA_DIRECTORY = "/home/lisa/Arupreza/UIDS-II/OtherLAB/Sonata"
checkpoint_to_evaluate = "/home/lisa/Arupreza/UIDS-II/SFTSrc/bert/TrainKia/mobilebert-can-attack-classifier-experimental"

print(f"\n--- Evaluating specific checkpoint: {checkpoint_to_evaluate} ---")
evaluate_model(
    model_path=checkpoint_to_evaluate,
    validation_data_directory=VALIDATION_DATA_DIRECTORY, # From your script's config
    time_gap=TIME_GAP_TEST                             # From your script's config
)


--- Evaluating specific checkpoint: /home/lisa/Arupreza/UIDS-II/SFTSrc/bert/TrainKia/mobilebert-can-attack-classifier-experimental ---

--- Starting Evaluation of Model: /home/lisa/Arupreza/UIDS-II/SFTSrc/bert/TrainKia/mobilebert-can-attack-classifier-experimental ---
Loading fine-tuned model and tokenizer...
Loading validation data from: /home/lisa/Arupreza/UIDS-II/OtherLAB/Sonata


Processing validation files: 100%|██████████| 4/4 [00:32<00:00,  8.01s/it]


Total validation segments loaded: 3702


Formatting segments: 100%|██████████| 3702/3702 [00:00<00:00, 7248.40it/s]


Running inference on validation data...


Inference: 100%|██████████| 232/232 [00:19<00:00, 11.95it/s]


Calculating metrics...

=== Validation Results ===
Accuracy:  0.9897
F1 Score:  0.9875
Precision: 1.0000
Recall:    0.9753

=== Confusion Matrix ===
               Predicted Negative | Predicted Positive
Actual Negative: 2161             | 0                
Actual Positive: 38               | 1503             


{'accuracy': 0.9897352782279849,
 'f1': 0.9875164257555847,
 'precision': 1.0,
 'recall': 0.9753406878650227}

In [11]:
# --- 10. Evaluate a Specific Checkpoint ---
# The path you provided is used here
TIME_GAP_TEST = 160.0
VALIDATION_DATA_DIRECTORY = "/home/lisa/Arupreza/UIDS-II/Split_data/OtherLAB/Forester"
checkpoint_to_evaluate = "/home/lisa/Arupreza/UIDS-II/SFTSrc/bert/TrainKia/mobilebert-can-attack-classifier-experimental"

print(f"\n--- Evaluating specific checkpoint: {checkpoint_to_evaluate} ---")
evaluate_model(
    model_path=checkpoint_to_evaluate,
    validation_data_directory=VALIDATION_DATA_DIRECTORY, # From your script's config
    time_gap=TIME_GAP_TEST                             # From your script's config
)


--- Evaluating specific checkpoint: /home/lisa/Arupreza/UIDS-II/SFTSrc/bert/TrainKia/mobilebert-can-attack-classifier-experimental ---

--- Starting Evaluation of Model: /home/lisa/Arupreza/UIDS-II/SFTSrc/bert/TrainKia/mobilebert-can-attack-classifier-experimental ---
Loading fine-tuned model and tokenizer...
Loading validation data from: /home/lisa/Arupreza/UIDS-II/Split_data/OtherLAB/Forester


Processing validation files: 100%|██████████| 6/6 [00:03<00:00,  1.89it/s]


Total validation segments loaded: 2770


Formatting segments: 100%|██████████| 2770/2770 [00:00<00:00, 8949.53it/s]


Running inference on validation data...


Inference: 100%|██████████| 174/174 [00:14<00:00, 12.10it/s]


Calculating metrics...

=== Validation Results ===
Accuracy:  0.9733
F1 Score:  0.9167
Precision: 0.8462
Recall:    1.0000

=== Confusion Matrix ===
               Predicted Negative | Predicted Positive
Actual Negative: 2289             | 74               
Actual Positive: 0                | 407              


{'accuracy': 0.9732851985559566,
 'f1': 0.9166666666666666,
 'precision': 0.8461538461538461,
 'recall': 1.0}

In [13]:
# --- 10. Evaluate a Specific Checkpoint ---
# The path you provided is used here
TIME_GAP_TEST = 100.0
VALIDATION_DATA_DIRECTORY = "/home/lisa/Arupreza/UIDS-II/Split_data/OtherLAB/Sil"
checkpoint_to_evaluate = "/home/lisa/Arupreza/UIDS-II/SFTSrc/bert/TrainKia/mobilebert-can-attack-classifier-experimental"

print(f"\n--- Evaluating specific checkpoint: {checkpoint_to_evaluate} ---")
evaluate_model(
    model_path=checkpoint_to_evaluate,
    validation_data_directory=VALIDATION_DATA_DIRECTORY, # From your script's config
    time_gap=TIME_GAP_TEST                             # From your script's config
)


--- Evaluating specific checkpoint: /home/lisa/Arupreza/UIDS-II/SFTSrc/bert/TrainKia/mobilebert-can-attack-classifier-experimental ---

--- Starting Evaluation of Model: /home/lisa/Arupreza/UIDS-II/SFTSrc/bert/TrainKia/mobilebert-can-attack-classifier-experimental ---
Loading fine-tuned model and tokenizer...
Loading validation data from: /home/lisa/Arupreza/UIDS-II/Split_data/OtherLAB/Sil


Processing validation files: 100%|██████████| 5/5 [00:04<00:00,  1.00it/s]


Total validation segments loaded: 4289


Formatting segments: 100%|██████████| 4289/4289 [00:00<00:00, 7637.92it/s]


Running inference on validation data...


Inference: 100%|██████████| 269/269 [00:22<00:00, 11.95it/s]


Calculating metrics...

=== Validation Results ===
Accuracy:  0.9564
F1 Score:  0.9530
Precision: 0.9102
Recall:    1.0000

=== Confusion Matrix ===
               Predicted Negative | Predicted Positive
Actual Negative: 2206             | 187              
Actual Positive: 0                | 1896             


{'accuracy': 0.9564000932618326,
 'f1': 0.9530032671525509,
 'precision': 0.9102256361017763,
 'recall': 1.0}

In [14]:
# --- 10. Evaluate a Specific Checkpoint ---
# The path you provided is used here
TIME_GAP_TEST = 105.0
VALIDATION_DATA_DIRECTORY = "/home/lisa/Arupreza/UIDS-II/Split_data/OtherLAB/Gen"
checkpoint_to_evaluate = "/home/lisa/Arupreza/UIDS-II/SFTSrc/bert/TrainKia/mobilebert-can-attack-classifier-experimental"

print(f"\n--- Evaluating specific checkpoint: {checkpoint_to_evaluate} ---")
evaluate_model(
    model_path=checkpoint_to_evaluate,
    validation_data_directory=VALIDATION_DATA_DIRECTORY, # From your script's config
    time_gap=TIME_GAP_TEST                             # From your script's config
)


--- Evaluating specific checkpoint: /home/lisa/Arupreza/UIDS-II/SFTSrc/bert/TrainKia/mobilebert-can-attack-classifier-experimental ---

--- Starting Evaluation of Model: /home/lisa/Arupreza/UIDS-II/SFTSrc/bert/TrainKia/mobilebert-can-attack-classifier-experimental ---
Loading fine-tuned model and tokenizer...
Loading validation data from: /home/lisa/Arupreza/UIDS-II/Split_data/OtherLAB/Gen


Processing validation files:   0%|          | 0/4 [00:00<?, ?it/s]/home/lisa/Arupreza/UIDS-II/SFTSrc/utils.py:91: DtypeWarning: Columns (29,31) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
Processing validation files:  50%|█████     | 2/4 [00:06<00:06,  3.16s/it]/home/lisa/Arupreza/UIDS-II/SFTSrc/utils.py:91: DtypeWarning: Columns (29,31) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
Processing validation files:  75%|███████▌  | 3/4 [00:10<00:03,  3.49s/it]/home/lisa/Arupreza/UIDS-II/SFTSrc/utils.py:91: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
Processing validation files: 100%|██████████| 4/4 [00:12<00:00,  3.19s/it]


Total validation segments loaded: 7232


Formatting segments: 100%|██████████| 7232/7232 [00:00<00:00, 7371.30it/s]


Running inference on validation data...


Inference: 100%|██████████| 452/452 [00:38<00:00, 11.85it/s]


Calculating metrics...

=== Validation Results ===
Accuracy:  0.9466
F1 Score:  0.9257
Precision: 0.8713
Recall:    0.9873

=== Confusion Matrix ===
               Predicted Negative | Predicted Positive
Actual Negative: 4443             | 355              
Actual Positive: 31               | 2403             


{'accuracy': 0.9466261061946902,
 'f1': 0.9256548536209553,
 'precision': 0.8712835387962291,
 'recall': 0.9872637633525062}

In [28]:
# --- 10. Evaluate a Specific Checkpoint ---
# The path you provided is used here
TIME_GAP_TEST = 95.0
checkpoint_to_evaluate = "/home/lisa/Arupreza/UIDS-II/SFTSrc/bert/TrainSil/mobilebert-can-attack-classifier-experimental"

In [29]:
VALIDATION_DATA_DIRECTORY = "/home/lisa/Arupreza/UIDS-II/Split_data/Test/Kia"
print(f"\n--- Evaluating specific checkpoint: {checkpoint_to_evaluate} ---")
evaluate_model(
    model_path=checkpoint_to_evaluate,
    validation_data_directory=VALIDATION_DATA_DIRECTORY, # From your script's config
    time_gap=TIME_GAP_TEST                             # From your script's config
)


--- Evaluating specific checkpoint: /home/lisa/Arupreza/UIDS-II/SFTSrc/bert/TrainSil/mobilebert-can-attack-classifier-experimental ---

--- Starting Evaluation of Model: /home/lisa/Arupreza/UIDS-II/SFTSrc/bert/TrainSil/mobilebert-can-attack-classifier-experimental ---
Loading fine-tuned model and tokenizer...
Loading validation data from: /home/lisa/Arupreza/UIDS-II/Split_data/Test/Kia


Processing validation files: 100%|██████████| 16/16 [00:08<00:00,  1.99it/s]


Total validation segments loaded: 7815


Formatting segments: 100%|██████████| 7815/7815 [00:00<00:00, 8545.35it/s]


Running inference on validation data...


Inference: 100%|██████████| 489/489 [00:40<00:00, 11.98it/s]


Calculating metrics...

=== Validation Results ===
Accuracy:  0.7239
F1 Score:  0.8336
Precision: 0.7154
Recall:    0.9985

=== Confusion Matrix ===
               Predicted Negative | Predicted Positive
Actual Negative: 252              | 2150             
Actual Positive: 8                | 5405             


{'accuracy': 0.7238643634037109,
 'f1': 0.8335903763109191,
 'precision': 0.7154202514890801,
 'recall': 0.998522076482542}

In [30]:
VALIDATION_DATA_DIRECTORY = "/home/lisa/Arupreza/UIDS-II/Split_data/Test/Sil"
print(f"\n--- Evaluating specific checkpoint: {checkpoint_to_evaluate} ---")
evaluate_model(
    model_path=checkpoint_to_evaluate,
    validation_data_directory=VALIDATION_DATA_DIRECTORY, # From your script's config
    time_gap=TIME_GAP_TEST                             # From your script's config
)


--- Evaluating specific checkpoint: /home/lisa/Arupreza/UIDS-II/SFTSrc/bert/TrainSil/mobilebert-can-attack-classifier-experimental ---

--- Starting Evaluation of Model: /home/lisa/Arupreza/UIDS-II/SFTSrc/bert/TrainSil/mobilebert-can-attack-classifier-experimental ---
Loading fine-tuned model and tokenizer...
Loading validation data from: /home/lisa/Arupreza/UIDS-II/Split_data/Test/Sil


Processing validation files: 100%|██████████| 16/16 [00:08<00:00,  1.96it/s]


Total validation segments loaded: 7992


Formatting segments: 100%|██████████| 7992/7992 [00:00<00:00, 8774.04it/s]


Running inference on validation data...


Inference: 100%|██████████| 500/500 [00:41<00:00, 11.91it/s]


Calculating metrics...

=== Validation Results ===
Accuracy:  0.8109
F1 Score:  0.8796
Precision: 0.7866
Recall:    0.9977

=== Confusion Matrix ===
               Predicted Negative | Predicted Positive
Actual Negative: 959              | 1498             
Actual Positive: 13               | 5522             


{'accuracy': 0.8109359359359359,
 'f1': 0.8796495420151335,
 'precision': 0.7866096866096867,
 'recall': 0.9976513098464318}

In [ ]:
print(f"\n--- Evaluating specific checkpoint: {checkpoint_to_evaluate} ---")
evaluate_model(
    model_path=checkpoint_to_evaluate,
    validation_data_directory=VALIDATION_DATA_DIRECTORY, # From your script's config
    time_gap=TIME_GAP_TEST                             # From your script's config
)

In [ ]:
print(f"\n--- Evaluating specific checkpoint: {checkpoint_to_evaluate} ---")
evaluate_model(
    model_path=checkpoint_to_evaluate,
    validation_data_directory=VALIDATION_DATA_DIRECTORY, # From your script's config
    time_gap=TIME_GAP_TEST                             # From your script's config
)